In [4]:
"""
====================================================================
  TRAFFIC DEMAND PREDICTION — FULL PIPELINE
  Best-accuracy version: LightGBM + XGBoost + CatBoost ensemble
  Metric: max(0, 100 * r2_score(actual, predicted))
====================================================================
HOW TO RUN:
  1. pip install lightgbm xgboost catboost pygeohash scikit-learn pandas numpy matplotlib seaborn
  2. Place train.csv, test.csv, sample_submission.csv in same folder
  3. python traffic_demand_prediction.py
  4. Upload submission.csv to the competition portal
====================================================================
"""

# ─────────────────────────── IMPORTS ────────────────────────────
import pandas as pd
import numpy as np
import warnings, os, time
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

try:
    import pygeohash as pgh
    GEO_OK = True
except ImportError:
    GEO_OK = False
    print("[WARN] pygeohash not found — lat/lon decoding disabled.")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

SEED     = 42
N_FOLDS  = 5
np.random.seed(SEED)
t0 = time.time()

In [5]:
!pip install lightgbm xgboost catboost pygeohash --quiet


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# ─────────────────────────── 1. LOAD DATA ───────────────────────
print("=" * 60)
print(" STEP 1: Loading data")
print("=" * 60)
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
print(f"  Train : {train.shape}   Test : {test.shape}")
print(f"  Target stats:\n{train['demand'].describe().round(3)}")

 STEP 1: Loading data
  Train : (77299, 11)   Test : (41778, 10)
  Target stats:
count    77299.000
mean         0.094
std          0.142
min          0.000
25%          0.018
50%          0.048
75%          0.109
max          1.000
Name: demand, dtype: float64


In [7]:
# ─────────────────────────── 2. EDA QUICK CHECKS ─────────────────
print("\n" + "=" * 60)
print(" STEP 2: Quick EDA")
print("=" * 60)
print("  Train dtypes:\n", train.dtypes.to_string())
print("\n  Missing (train):\n", train.isnull().sum().to_string())
print("\n  Missing (test):\n",  test.isnull().sum().to_string())

# Save demand distribution chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train['demand'].hist(bins=60, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title("Demand Distribution", fontsize=13)
axes[0].set_xlabel("demand")
np.log1p(train['demand']).hist(bins=60, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title("log1p(Demand) Distribution", fontsize=13)
axes[1].set_xlabel("log1p(demand)")
plt.tight_layout()
plt.savefig("eda_demand_dist.png", dpi=150)
plt.close()
print("  eda_demand_dist.png saved.")


 STEP 2: Quick EDA
  Train dtypes:
 Index              int64
geohash           object
day                int64
timestamp         object
demand           float64
RoadType          object
NumberofLanes      int64
LargeVehicles     object
Landmarks         object
Temperature      float64
Weather           object

  Missing (train):
 Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797

  Missing (test):
 Index               0
geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
  eda_demand_dist.png saved.


In [8]:
# ─────────────────────────── 3. FEATURE ENGINEERING ─────────────
print("\n" + "=" * 60)
print(" STEP 3: Feature Engineering")
print("=" * 60)

DAY_MAP = {d: i for i, d in enumerate(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])}

def decode_geo(gh):
    try:
        lat, lon = pgh.decode(str(gh))
        return lat, lon
    except:
        return np.nan, np.nan

def feature_engineer(df, geo_map=None, encoders=None, is_train=True):
    df = df.copy()

    # ── 3A. GEOHASH → lat/lon + spatial hierarchy prefixes ─────────
    if GEO_OK:
        if geo_map is None:
            unique_gh = df['geohash'].dropna().unique()
            geo_map   = {gh: decode_geo(gh) for gh in unique_gh}
        df['lat'] = df['geohash'].map(lambda x: geo_map.get(x, (np.nan, np.nan))[0])
        df['lon'] = df['geohash'].map(lambda x: geo_map.get(x, (np.nan, np.nan))[1])
    else:
        df['lat'] = np.nan
        df['lon'] = np.nan

    # Spatial hierarchy (coarse → fine)
    df['geo3'] = df['geohash'].astype(str).str[:3]  # ~156 km²
    df['geo4'] = df['geohash'].astype(str).str[:4]  # ~20 km²
    df['geo5'] = df['geohash'].astype(str).str[:5]  # ~2.4 km²
    df['geo6'] = df['geohash'].astype(str).str[:6]  # full hash

    # ── 3B. TIMESTAMP features ──────────────────────────────────────
    ts = pd.to_datetime(df['timestamp'], infer_datetime_format=True, errors='coerce')
    df['hour']        = ts.dt.hour
    df['minute']      = ts.dt.minute
    df['hhmm']        = df['hour'] * 60 + df['minute']   # minute-of-day (0-1439)
    df['is_peak_am']  = df['hour'].between(7, 9).astype(int)
    df['is_peak_pm']  = df['hour'].between(17, 19).astype(int)
    df['is_night']    = (~df['hour'].between(6, 22)).astype(int)
    df['is_midday']   = df['hour'].between(11, 13).astype(int)
    # Cyclical hour encoding (so hour 23 ≈ hour 0)
    df['hour_sin']    = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']    = np.cos(2 * np.pi * df['hour'] / 24)
    df['hhmm_sin']    = np.sin(2 * np.pi * df['hhmm'] / 1440)
    df['hhmm_cos']    = np.cos(2 * np.pi * df['hhmm'] / 1440)

    # ── 3C. DAY-OF-WEEK features ────────────────────────────────────
    df['day_num']    = df['day'].map(DAY_MAP).fillna(-1).astype(int)
    df['is_weekend'] = df['day_num'].isin([5, 6]).astype(int)
    df['day_sin']    = np.sin(2 * np.pi * df['day_num'] / 7)
    df['day_cos']    = np.cos(2 * np.pi * df['day_num'] / 7)

    # ── 3D. INTERACTION FEATURES ────────────────────────────────────
    # Joint effects that trees can learn more easily as explicit keys
    df['geo4_hour']     = df['geo4'] + '_h' + df['hour'].astype(str)
    df['geo4_day']      = df['geo4'] + '_d' + df['day_num'].astype(str)
    df['day_hour']      = 'd' + df['day_num'].astype(str) + '_h' + df['hour'].astype(str)
    df['geo4_peak_am']  = df['geo4'] + '_pam' + df['is_peak_am'].astype(str)
    df['geo4_peak_pm']  = df['geo4'] + '_ppm' + df['is_peak_pm'].astype(str)
    df['geo4_weekend']  = df['geo4'] + '_we'  + df['is_weekend'].astype(str)

    # ── 3E. ROAD INFRASTRUCTURE ─────────────────────────────────────
    df['NumberofLanes'] = pd.to_numeric(df['NumberofLanes'], errors='coerce').fillna(0)
    df['high_capacity'] = (df['NumberofLanes'] >= 4).astype(int)
    df['single_lane']   = (df['NumberofLanes'] == 1).astype(int)

    for col in ['LargeVehicles', 'Landmarks']:
        if col in df.columns:
            df[col] = df[col].map(
                lambda x: 1 if str(x).strip().lower() in ('yes','1','true','y') else 0)

    # ── 3F. WEATHER / TEMPERATURE ────────────────────────────────────
    df['Temperature']   = pd.to_numeric(df['Temperature'], errors='coerce')
    df['temp_missing']  = df['Temperature'].isnull().astype(int)
    med_temp            = df['Temperature'].median()
    df['Temperature'].fillna(med_temp, inplace=True)
    df['temp_high']     = (df['Temperature'] > df['Temperature'].quantile(0.75)).astype(int)
    df['temp_low']      = (df['Temperature'] < df['Temperature'].quantile(0.25)).astype(int)

    # ── 3G. LABEL ENCODING of all object cols ────────────────────────
    cat_cols = ['RoadType','Weather',
                'geo3','geo4','geo5','geo6',
                'geo4_hour','geo4_day','day_hour',
                'geo4_peak_am','geo4_peak_pm','geo4_weekend']

    if encoders is None:
        encoders = {}
    for col in cat_cols:
        if col not in df.columns:
            continue
        df[col] = df[col].astype(str).fillna('MISSING')
        if col not in encoders:
            le = LabelEncoder()
            df[col + '_enc'] = le.fit_transform(df[col])
            encoders[col] = le
        else:
            le   = encoders[col]
            known = set(le.classes_)
            df[col] = df[col].map(lambda x: x if x in known else 'MISSING')
            if 'MISSING' not in known:
                le.classes_ = np.append(le.classes_, 'MISSING')
            df[col + '_enc'] = le.transform(df[col])

    return df, geo_map, encoders


train_eng, geo_map, encoders = feature_engineer(train, is_train=True)
test_eng,  _,       _        = feature_engineer(test,  geo_map=geo_map,
                                                 encoders=encoders, is_train=False)
print(f"  After base FE  → train: {train_eng.shape}, test: {test_eng.shape}")



 STEP 3: Feature Engineering
  After base FE  → train: (77299, 55), test: (41778, 54)


In [9]:
# ─────────────────────────── 4. TARGET AGGREGATION FEATURES ──────
print("\n" + "=" * 60)
print(" STEP 4: Target Aggregation Features")
print("=" * 60)

def target_agg(tr, te, grp_col, tgt='demand', prefix=None):
    """Mean / median / std of target per group — computed on train only."""
    prefix = prefix or grp_col
    stats = tr.groupby(grp_col)[tgt].agg(
        **{f'{prefix}_mean':   'mean',
           f'{prefix}_median': 'median',
           f'{prefix}_std':    'std',
           f'{prefix}_count':  'count'}
    ).reset_index()
    tr2 = tr.merge(stats, on=grp_col, how='left')
    te2 = te.merge(stats, on=grp_col, how='left')
    g_mean   = tr[tgt].mean()
    g_median = tr[tgt].median()
    g_std    = tr[tgt].std()
    for c in stats.columns[1:]:
        fv = g_mean if 'mean' in c else g_median if 'median' in c \
             else g_std if 'std' in c else 1
        tr2[c].fillna(fv, inplace=True)
        te2[c].fillna(fv, inplace=True)
    return tr2, te2

AGG_GROUPS = [
    ('geo3',        'geo3'),
    ('geo4',        'geo4'),
    ('geo5',        'geo5'),
    ('geo6_enc',    'geo6'),
    ('hour',        'hour'),
    ('day_num',     'day'),
    ('day_hour',    'day_hour'),
    ('geo4_hour',   'geo4_hour'),
    ('geo4_day',    'geo4_day'),
    ('RoadType_enc','road'),
    ('Weather_enc', 'weather'),
]

for grp_col, prefix in AGG_GROUPS:
    if grp_col in train_eng.columns:
        train_eng, test_eng = target_agg(train_eng, test_eng, grp_col, prefix=prefix)
        print(f"  Added agg features for: {grp_col}")

print(f"  Final → train: {train_eng.shape}, test: {test_eng.shape}")



 STEP 4: Target Aggregation Features
  Added agg features for: geo3
  Added agg features for: geo4
  Added agg features for: geo5
  Added agg features for: geo6_enc
  Added agg features for: hour
  Added agg features for: day_num
  Added agg features for: day_hour
  Added agg features for: geo4_hour
  Added agg features for: geo4_day
  Added agg features for: RoadType_enc
  Added agg features for: Weather_enc
  Final → train: (77299, 99), test: (41778, 98)


In [10]:
# ─────────────────────────── 5. PREPARE MODEL INPUT ──────────────
print("\n" + "=" * 60)
print(" STEP 5: Preparing X / y")
print("=" * 60)

DROP = ['Index','geohash','day','timestamp',
        'geo3','geo4','geo5','geo6',
        'geo4_hour','geo4_day','day_hour',
        'geo4_peak_am','geo4_peak_pm','geo4_weekend',
        'RoadType','Weather','demand']

TARGET = 'demand'
feat_cols = [c for c in train_eng.columns
             if c not in DROP and train_eng[c].dtype != object]

X      = train_eng[feat_cols].copy().fillna(-999)
y      = train_eng[TARGET].copy()
X_test = test_eng[feat_cols].copy().fillna(-999)

print(f"  Feature count : {len(feat_cols)}")
print(f"  X shape       : {X.shape}")
print(f"  X_test shape  : {X_test.shape}")
print(f"  Feature list  : {feat_cols}")


 STEP 5: Preparing X / y
  Feature count : 82
  X shape       : (77299, 82)
  X_test shape  : (41778, 82)
  Feature list  : ['NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'lat', 'lon', 'hour', 'minute', 'hhmm', 'is_peak_am', 'is_peak_pm', 'is_night', 'is_midday', 'hour_sin', 'hour_cos', 'hhmm_sin', 'hhmm_cos', 'day_num', 'is_weekend', 'day_sin', 'day_cos', 'high_capacity', 'single_lane', 'temp_missing', 'temp_high', 'temp_low', 'RoadType_enc', 'Weather_enc', 'geo3_enc', 'geo4_enc', 'geo5_enc', 'geo6_enc', 'geo4_hour_enc', 'geo4_day_enc', 'day_hour_enc', 'geo4_peak_am_enc', 'geo4_peak_pm_enc', 'geo4_weekend_enc', 'geo3_mean', 'geo3_median', 'geo3_std', 'geo3_count', 'geo4_mean', 'geo4_median', 'geo4_std', 'geo4_count', 'geo5_mean', 'geo5_median', 'geo5_std', 'geo5_count', 'geo6_mean', 'geo6_median', 'geo6_std', 'geo6_count', 'hour_mean', 'hour_median', 'hour_std', 'hour_count', 'day_mean', 'day_median', 'day_std', 'day_count', 'day_hour_mean', 'day_hour_median', 'day_ho

In [13]:
# ─────────────────────────── 6. MODEL TRAINING ───────────────────
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# ── 6A. LightGBM ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" STEP 6A: LightGBM")
print("=" * 60)

lgb_params = dict(
    objective       = 'regression',
    metric          = 'rmse',
    n_estimators    = 3000,
    learning_rate   = 0.02,
    num_leaves      = 255,
    max_depth       = -1,
    min_child_samples = 15,
    feature_fraction  = 0.75,
    bagging_fraction  = 0.75,
    bagging_freq      = 5,
    reg_alpha         = 0.05,
    reg_lambda        = 0.1,
    min_split_gain    = 0.001,
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1,
)

oof_lgb  = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))
lgb_model_last = None

for fold, (tr_i, val_i) in enumerate(kf.split(X)):
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(
        X.iloc[tr_i], y.iloc[tr_i],
        eval_set     = [(X.iloc[val_i], y.iloc[val_i])],
        callbacks    = [lgb.early_stopping(150, verbose=False),
                        lgb.log_evaluation(period=-1)]
    )
    oof_lgb[val_i]  = m.predict(X.iloc[val_i])
    test_lgb       += m.predict(X_test) / N_FOLDS
    lgb_model_last  = m
    sc = max(0, 100 * r2_score(y.iloc[val_i], oof_lgb[val_i]))
    print(f"  Fold {fold+1}: Competition Score = {sc:.3f}  (best_iter={m.best_iteration_})")

lgb_score = max(0, 100 * r2_score(y, oof_lgb))
print(f"\n  ★ LightGBM OOF Score : {lgb_score:.3f}")

# Feature importance chart
fi = pd.DataFrame({'feature': feat_cols,
                   'importance': lgb_model_last.feature_importances_})
fi.sort_values('importance', ascending=False, inplace=True)
plt.figure(figsize=(10, 9))
sns.barplot(data=fi.head(30), x='importance', y='feature', palette='viridis')
plt.title('Top-30 Feature Importances — LightGBM', fontsize=13)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.close()
print("  feature_importance.png saved.")

# ── 6B. XGBoost ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" STEP 6B: XGBoost")
print("=" * 60)

xgb_params = dict(
    n_estimators    = 3000,
    learning_rate   = 0.02,
    max_depth       = 8,
    min_child_weight = 5,
    subsample        = 0.75,
    colsample_bytree = 0.75,
    reg_alpha        = 0.05,
    reg_lambda       = 1.0,
    random_state     = SEED,
    n_jobs           = -1,
    verbosity        = 0,
    tree_method      = 'hist',
    early_stopping_rounds = 150,
)

oof_xgb  = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

for fold, (tr_i, val_i) in enumerate(kf.split(X)):
    xm = xgb.XGBRegressor(**xgb_params)
    xm.fit(
        X.iloc[tr_i], y.iloc[tr_i],
        eval_set           = [(X.iloc[val_i], y.iloc[val_i])],
        verbose            = False,
    )
    oof_xgb[val_i]  = xm.predict(X.iloc[val_i])
    test_xgb       += xm.predict(X_test) / N_FOLDS
    sc = max(0, 100 * r2_score(y.iloc[val_i], oof_xgb[val_i]))
    print(f"  Fold {fold+1}: Competition Score = {sc:.3f}  (best_iter={xm.best_iteration})")

xgb_score = max(0, 100 * r2_score(y, oof_xgb))
print(f"\n  ★ XGBoost OOF Score : {xgb_score:.3f}")

# ── 6C. CatBoost ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" STEP 6C: CatBoost")
print("=" * 60)

cat_params = dict(
    iterations        = 3000,
    learning_rate     = 0.02,
    depth             = 8,
    l2_leaf_reg       = 3,
    random_strength   = 1,
    bagging_temperature = 0.5,
    od_type           = 'Iter',
    od_wait           = 150,
    random_seed       = SEED,
    verbose           = False,
    task_type         = 'CPU',
)

oof_cat  = np.zeros(len(X))
test_cat = np.zeros(len(X_test))

for fold, (tr_i, val_i) in enumerate(kf.split(X)):
    cm = CatBoostRegressor(**cat_params)
    cm.fit(
        X.iloc[tr_i], y.iloc[tr_i],
        eval_set   = (X.iloc[val_i], y.iloc[val_i]),
        use_best_model = True,
    )
    oof_cat[val_i]  = cm.predict(X.iloc[val_i])
    test_cat       += cm.predict(X_test) / N_FOLDS
    sc = max(0, 100 * r2_score(y.iloc[val_i], oof_cat[val_i]))
    print(f"  Fold {fold+1}: Competition Score = {sc:.3f}")

cat_score = max(0, 100 * r2_score(y, oof_cat))
print(f"\n  ★ CatBoost OOF Score : {cat_score:.3f}")



 STEP 6A: LightGBM
  Fold 1: Competition Score = 92.701  (best_iter=374)
  Fold 2: Competition Score = 93.205  (best_iter=789)
  Fold 3: Competition Score = 92.857  (best_iter=395)
  Fold 4: Competition Score = 92.162  (best_iter=430)
  Fold 5: Competition Score = 92.414  (best_iter=275)

  ★ LightGBM OOF Score : 92.677
  feature_importance.png saved.

 STEP 6B: XGBoost
  Fold 1: Competition Score = 92.857  (best_iter=515)
  Fold 2: Competition Score = 93.229  (best_iter=672)
  Fold 3: Competition Score = 93.004  (best_iter=420)
  Fold 4: Competition Score = 92.202  (best_iter=494)
  Fold 5: Competition Score = 92.564  (best_iter=423)

  ★ XGBoost OOF Score : 92.780

 STEP 6C: CatBoost
  Fold 1: Competition Score = 93.043
  Fold 2: Competition Score = 93.325
  Fold 3: Competition Score = 93.005
  Fold 4: Competition Score = 92.328
  Fold 5: Competition Score = 92.651

  ★ CatBoost OOF Score : 92.879


In [14]:
# ─────────────────────────── 7. OPTIMAL BLEND ────────────────────
print("\n" + "=" * 60)
print(" STEP 7: Optimal Weighted Blend")
print("=" * 60)

# Score-proportional weights
total = lgb_score + xgb_score + cat_score
w_lgb = lgb_score / total
w_xgb = xgb_score / total
w_cat = cat_score / total
print(f"  Weights → LGB: {w_lgb:.3f}  XGB: {w_xgb:.3f}  CAT: {w_cat:.3f}")

oof_blend  = w_lgb * oof_lgb  + w_xgb * oof_xgb  + w_cat * oof_cat
test_blend = w_lgb * test_lgb + w_xgb * test_xgb + w_cat * test_cat

blend_score = max(0, 100 * r2_score(y, oof_blend))
print(f"\n  ★ BLEND OOF Score : {blend_score:.3f}")

print(f"\n  Summary:")
print(f"    LightGBM : {lgb_score:.3f}")
print(f"    XGBoost  : {xgb_score:.3f}")
print(f"    CatBoost : {cat_score:.3f}")
print(f"    BLEND    : {blend_score:.3f}")

# Clip to non-negative (demand cannot be negative)
test_final = np.clip(test_blend, 0, None)


 STEP 7: Optimal Weighted Blend
  Weights → LGB: 0.333  XGB: 0.333  CAT: 0.334

  ★ BLEND OOF Score : 92.880

  Summary:
    LightGBM : 92.677
    XGBoost  : 92.780
    CatBoost : 92.879
    BLEND    : 92.880


In [15]:
# ─────────────────────────── 8. OOF DIAGNOSTICS PLOT ─────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y, oof_blend, alpha=0.15, s=4, color='steelblue')
mn, mx = y.min(), y.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect')
axes[0].set_xlabel('Actual demand')
axes[0].set_ylabel('Predicted demand')
axes[0].set_title(f'OOF: Actual vs Predicted  (score={blend_score:.2f})')
axes[0].legend()

residuals = y - oof_blend
axes[1].hist(residuals, bins=80, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Residual (actual − predicted)')
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('oof_diagnostics.png', dpi=150)
plt.close()
print("\n  oof_diagnostics.png saved.")


  oof_diagnostics.png saved.


In [16]:
# ─────────────────────────── 9. SUBMISSION ───────────────────────
print("\n" + "=" * 60)
print(" STEP 9: Generating Submission")
print("=" * 60)

submission = pd.DataFrame({
    'Index' : test['Index'],
    'demand': test_final
})
submission.to_csv('submission.csv', index=False)
print(f"  submission.csv saved — shape: {submission.shape}")
print(submission.head(10).to_string(index=False))

# Sanity checks
assert submission.shape[0] == 41778,            "Row count mismatch!"
assert list(submission.columns) == ['Index','demand'], "Column name mismatch!"
assert submission['demand'].isnull().sum() == 0,"NaN in predictions!"
print("\n  ✓ All sanity checks passed.")
print(f"\n  Total runtime: {(time.time()-t0)/60:.1f} min")
print("=" * 60)


 STEP 9: Generating Submission
  submission.csv saved — shape: (41778, 2)
 Index   demand
     0 0.054099
     1 0.035656
     2 0.015107
     3 0.048134
     4 0.064889
     5 0.020211
     6 0.027783
     7 0.084749
     8 0.030478
     9 0.083752

  ✓ All sanity checks passed.

  Total runtime: 37.3 min
